# Reliance Industries — Stock Price Analysis & 30-Day Out-of-Sample Forecasting
## Production ML Pipeline with MASE, Residual Bootstrap Intervals & Multi-Origin Backtest

**Executive Overview:**
Enterprise-grade stock price forecasting system for Reliance Industries (`RELIANCE.NS`).
- **Data Scope**: Historical OHLCV data (2020-10-19 to 2023-10-16).
- **Data Preprocessing**: Business-day resampling, gap imputation, and rolling Z-score outlier capping.
- **Stationarity**: Log-differencing transformation & Augmented Dickey-Fuller (ADF) test.
- **Validation**: 5-Fold Expanding Window Cross-Validation + Multi-Origin Rolling 30-Day Backtest + 2023 Hold-Out set.
- **Models**: Naive Baseline, SARIMA (0,1,0), Exponential Smoothing (ETS), Prophet, State Space, XGBoost, LightGBM, CatBoost (8 Models total).
- **Metrics**: RMSE, MAE, MAPE, sMAPE, **MASE** (Mean Absolute Scaled Error), Directional Accuracy, $R^2$.
- **Intervals**: Residual-bootstrap prediction intervals for tree models + model-native CIs for statistical models.
- **Interpretability**: SHAP (SHapley Additive exPlanations) TreeExplainer feature attributions.


## Phase 1: Setup & Open-Source Data Ingestion
Import `src` package utilities, fetch historical OHLCV data, apply business-day resampling, gap imputation, and rolling Z-score outlier capping.


In [ ]:
import os
import sys
if '.' not in sys.path:
    sys.path.insert(0, '.')

import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import matplotlib
matplotlib.use('Agg')

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid")

from src.data import fetch_data, resample_to_daily, impute_missing_values, detect_outliers_rolling_zscore, make_stationary, detect_concept_drift
from src.features import build_advanced_features, create_external_features
from src.models import train_xgboost, train_lightgbm, train_catboost, train_arima, train_ets, train_prophet, train_state_space
from src.evaluation import calculate_mase, evaluate_forecast, residual_bootstrap_intervals, rolling_30day_backtest, expanding_window_cv
from src.forecasting import generate_recursive_forecast

# Fetch raw dataset
df_raw = fetch_data(ticker="RELIANCE.NS", start_date="2020-10-19")
print(f"Raw Data Ingested: {len(df_raw)} records from {df_raw.index.min().strftime('%Y-%m-%d')} to {df_raw.index.max().strftime('%Y-%m-%d')}")

# Resample to business days & impute missing values
df_daily = resample_to_daily(df_raw)
df_clean = impute_missing_values(df_daily, method='ffill', max_gap=5)

# Detect and cap extreme outliers using rolling Z-score
df_capped, outlier_mask = detect_outliers_rolling_zscore(df_clean, column='Close', window=20, threshold=3.0)

df = df_capped.reset_index()
df["Date"] = pd.to_datetime(df["Date"]).dt.floor('D')
df = df.sort_values("Date").reset_index(drop=True)

print(f"Preprocessed Dataset: {len(df)} records ({outlier_mask.sum()} outliers capped).")
print(df.head())


## Phase 2: Exploratory Data Analysis & Integrity Audits
Perform missing value checks, duplicate date verification, OHLC logical price consistency checks, and train/test split.


In [ ]:
# Data Integrity Audits
print("=== Missing Values Audit ===")
print(df.isnull().sum())

print("\n=== Duplicate Dates Audit ===")
print("Duplicates:", df["Date"].duplicated().sum())

print("\n=== OHLC Consistency Check ===")
print("High < Open :", (df["High"] < df["Open"]).sum())
print("High < Close:", (df["High"] < df["Close"]).sum())
print("Low > Open  :", (df["Low"] > df["Open"]).sum())
print("Low > Close :", (df["Low"] > df["Close"]).sum())


In [ ]:
# Chronological Train-Test Split (2020-2022 Train, 2023 Test)
train = df[df["Date"] < "2023-01-01"].copy()
test = df[df["Date"] >= "2023-01-01"].copy()

print(f"Training Set : {len(train)} rows ({train['Date'].min().strftime('%Y-%m-%d')} to {train['Date'].max().strftime('%Y-%m-%d')})")
print(f"Testing Set  : {len(test)} rows ({test['Date'].min().strftime('%Y-%m-%d')} to {test['Date'].max().strftime('%Y-%m-%d')})")

plt.figure(figsize=(14, 5))
plt.plot(train["Date"], train["Close"], label="Training Set (2020–2022)", color="#004c99")
plt.plot(test["Date"], test["Close"], label="Testing Set (2023 Hold-out)", color="#e67e22")
plt.title("Reliance Industries — Chronological Train/Test Split", fontsize=14, fontweight="bold")
plt.ylabel("Price (INR)")
plt.legend()
plt.tight_layout()
plt.show()


## Phase 3: Stationarity Analysis & Log-Differencing
Test for stationarity using the Augmented Dickey-Fuller (ADF) test on raw prices vs log-differenced series.


In [ ]:
from statsmodels.tsa.stattools import adfuller

print("=== Augmented Dickey-Fuller (ADF) Test ===")
result_raw = adfuller(train["Close"])
print(f"Raw Series ADF Statistic: {result_raw[0]:.4f}")
print(f"p-value: {result_raw[1]:.4f} ({'Stationary' if result_raw[1] < 0.05 else 'Non-Stationary'})")

df_stat = make_stationary(train, column='Close', method='log_diff')
result_diff = adfuller(df_stat["Close_stationary"])
print(f"\nLog-Difference ADF Statistic: {result_diff[0]:.4f}")
print(f"p-value: {result_diff[1]:.4f} ({'Stationary' if result_diff[1] < 0.05 else 'Non-Stationary'})")
print("Conclusion: Log-differenced series is stationary (d = 1 for SARIMA).")


## Phase 4: Leak-Free Technical Indicator Suite
Construct 21 technical indicators (**RSI-14**, **MACD**, **Bollinger Width**, **ATR-14**, **ROC-10**, **Lags 1..10**, **Rolling Stats**). Every indicator is shifted by 1 step (`shift(1)`) to ensure **zero data leakage**.


In [ ]:
df_ml = build_advanced_features(df)
feature_cols = [c for c in df_ml.columns if c not in ['Date', 'Close', 'High', 'Low', 'Volume']]

print(f"Engineered {len(feature_cols)} technical features.")
print(f"Feature set shape: {df_ml.shape}")
print(df_ml[feature_cols].head())


## Phase 5: External Macro Factors Integration
Fetch market benchmark data for **Crude Oil** (`CL=F`), **Nifty 50 Index** (`^NSEI`), and **USD/INR Exchange Rate** (`INR=X`) and compute cross-asset price correlations.


In [ ]:
ext_tickers = {'Crude_Oil': 'CL=F', 'NIFTY50': '^NSEI', 'USD_INR': 'INR=X'}
ext_dfs = {}

import yfinance as yf
for name, ticker in ext_tickers.items():
    try:
        d_raw = yf.download(ticker, start="2020-10-19", end="2023-10-17", progress=False)
        if isinstance(d_raw.columns, pd.MultiIndex):
            c_val = d_raw['Adj Close'][ticker].values if 'Adj Close' in d_raw else d_raw['Close'][ticker].values
        else:
            c_val = d_raw['Adj Close'].values if 'Adj Close' in d_raw else d_raw['Close'].values
        ext_dfs[name] = pd.DataFrame({'Date': pd.to_datetime(d_raw.index.date), f'{name}_Close': c_val})
    except Exception as e:
        print(f"Could not download {name}: {e}")

df_macro = df[['Date', 'Close']].copy()
df_macro['Date'] = pd.to_datetime(df_macro['Date']).astype('datetime64[ns]')
for name, d_ext in ext_dfs.items():
    d_ext['Date'] = pd.to_datetime(d_ext['Date']).astype('datetime64[ns]')
    df_macro = pd.merge_asof(df_macro.sort_values('Date'), d_ext.sort_values('Date'), on='Date', direction='nearest')

print("=== Macro Correlation Matrix ===")
corr_matrix = df_macro.drop(columns=['Date']).corr()
print(corr_matrix)

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", fmt=".2f", linewidths=0.5)
plt.title("Cross-Asset Price Correlation Matrix", fontweight="bold")
plt.tight_layout()
plt.show()


## Phase 6: 5-Fold Expanding Window Walk-Forward Validation (WFV)
Evaluate tree-based models across expanding time-series folds using `TimeSeriesSplit`.


In [ ]:
wfv_train = df_ml[df_ml['Date'] < '2023-01-01'].reset_index(drop=True)
X_wfv = wfv_train[feature_cols]
y_wfv = wfv_train['Close']

from xgboost import XGBRegressor

xgb_cv_results = expanding_window_cv(
    lambda: XGBRegressor(n_estimators=300, max_depth=4, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, random_state=42, verbosity=0),
    X_wfv, y_wfv, n_splits=5
)
print("=== 5-Fold Expanding Window CV Results (XGBoost) ===")
print(xgb_cv_results[['Model', 'RMSE', 'MAE', 'MAPE', 'sMAPE', 'MASE', 'Directional_Accuracy']])


## Phase 7: Model Benchmarking on 2023 Hold-Out Test Set (with MASE)
Train 8 distinct models and evaluate metrics (RMSE, MAE, MAPE, sMAPE, **MASE**, Directional Accuracy, $R^2$) passing `y_train`.


In [ ]:
feature_train = df_ml[df_ml["Date"] < "2023-01-01"].copy()
feature_test = df_ml[df_ml["Date"] >= "2023-01-01"].copy()

X_train, y_train = feature_train[feature_cols], feature_train["Close"]
X_test, y_test = feature_test[feature_cols], feature_test["Close"]

# Fit Tree Models
xgb = train_xgboost(X_train, y_train)
lgb = train_lightgbm(X_train, y_train)
cat = train_catboost(X_train, y_train)

# Fit Statistical Models
train_close = train.set_index('Date')['Close']
test_close = test.set_index('Date')['Close']

naive_pred = pd.Series([train_close.iloc[-1]] * len(test_close), index=test_close.index)
sarima_fit = train_arima(train_close, order=(0, 1, 0))
sarima_pred = sarima_fit.get_forecast(steps=len(test_close)).predicted_mean

ets_fit = train_ets(train_close)
ets_pred = ets_fit.forecast(len(test_close))

prophet_fit = train_prophet(train_close, train['Date'])
fut_p = pd.DataFrame({'ds': test['Date']})
prophet_pred = prophet_fit.predict(fut_p)['yhat'].values

ss_fit = train_state_space(train_close)
ss_pred = ss_fit.get_forecast(steps=len(test_close)).predicted_mean

test_metrics = [
    evaluate_forecast(test_close.values, naive_pred.values, "Naive Baseline", y_train=train_close.values),
    evaluate_forecast(test_close.values, sarima_pred.values, "SARIMA (0,1,0)", y_train=train_close.values),
    evaluate_forecast(test_close.values, ets_pred.values, "Exponential Smoothing (ETS)", y_train=train_close.values),
    evaluate_forecast(test_close.values, prophet_pred, "Prophet (No Seasonality)", y_train=train_close.values),
    evaluate_forecast(test_close.values, ss_pred.values, "State Space (Structural)", y_train=train_close.values),
    evaluate_forecast(y_test.values, xgb.predict(X_test), "XGBoost", y_train=y_train.values),
    evaluate_forecast(y_test.values, lgb.predict(X_test), "LightGBM", y_train=y_train.values),
    evaluate_forecast(y_test.values, cat.predict(X_test), "CatBoost", y_train=y_train.values)
]

comparison_df = pd.DataFrame(test_metrics).set_index("Model")
print("=== Model Performance Benchmark (Held-Out 2023 Test Set with MASE) ===")
print(comparison_df)

plt.figure(figsize=(14, 6))
plt.plot(feature_test['Date'], y_test.values, label="Actual Test Price", color="black", lw=2)
plt.plot(feature_test['Date'], xgb.predict(X_test), label="XGBoost", color="green")
plt.plot(feature_test['Date'], lgb.predict(X_test), label="LightGBM", color="blue")
plt.plot(feature_test['Date'], cat.predict(X_test), label="CatBoost", color="red")
plt.plot(feature_test['Date'], prophet_pred, label="Prophet", color="orange", ls="--")
plt.title("Model Forecast Benchmarking vs Actual Price (2023 Test Period)", fontsize=14, fontweight="bold")
plt.ylabel("Price (INR)")
plt.legend()
plt.grid(True)
plt.show()


## Phase 8: Prediction Intervals for Tree-Based Models
Compute residual bootstrap prediction intervals (`residual_bootstrap_intervals`) for XGBoost, LightGBM, and CatBoost 30-day recursive forecasts.

*Note: Empirical interval from residual resampling — not a model-native confidence interval.*


In [ ]:
# Compute residuals on test set
xgb_res = y_test.to_numpy() - xgb.predict(X_test)
lgb_res = y_test.to_numpy() - lgb.predict(X_test)
cat_res = y_test.to_numpy() - cat.predict(X_test)

# Generate 30-day recursive forecasts with residual bootstrap intervals
xgb_forecast_30 = generate_recursive_forecast(xgb, df_ml, feature_cols, horizon=30, residuals=xgb_res)
lgbm_forecast_30 = generate_recursive_forecast(lgb, df_ml, feature_cols, horizon=30, residuals=lgb_res)
cat_forecast_30 = generate_recursive_forecast(cat, df_ml, feature_cols, horizon=30, residuals=cat_res)

print("=== 30-Day Recursive Forecasts with Residual Bootstrap Intervals ===")
print("XGBoost 30-Day Head:")
print(xgb_forecast_30.head())

plt.figure(figsize=(14, 6))
plt.plot(xgb_forecast_30['Date'], xgb_forecast_30['Predicted_Close'], label="XGBoost Forecast", color="green", lw=2, ls="--")
plt.fill_between(xgb_forecast_30['Date'], xgb_forecast_30['Lower_95'], xgb_forecast_30['Upper_95'], color="green", alpha=0.15, label="XGBoost 95% Bootstrap Band")

plt.plot(lgbm_forecast_30['Date'], lgbm_forecast_30['Predicted_Close'], label="LightGBM Forecast", color="blue", lw=2, ls="--")
plt.fill_between(lgbm_forecast_30['Date'], lgbm_forecast_30['Lower_95'], lgbm_forecast_30['Upper_95'], color="blue", alpha=0.15, label="LightGBM 95% Bootstrap Band")

plt.title("Tree Models 30-Day Out-of-Sample Forecasts with Residual-Bootstrap Intervals", fontsize=14, fontweight="bold")
plt.ylabel("Price (INR)")
plt.legend()
plt.grid(True)
plt.show()
print("Empirical interval from residual resampling — not a model-native confidence interval.")


## Phase 9: Multi-Origin Rolling 30-Day Backtest
Evaluate all 8 models across multiple historical 30-day forecast origins on `df["Close"]`, refitting models at each origin.


In [ ]:
# Prepare full series for rolling backtest
full_series = df.set_index('Date')['Close']
initial_train_size = len(train)  # ~575 trading days

def forecast_naive(train_slice, horizon):
    return np.full(horizon, train_slice.iloc[-1])

def forecast_sarima(train_slice, horizon):
    fit = train_arima(train_slice, order=(0, 1, 0))
    return fit.get_forecast(steps=horizon).predicted_mean.values

def forecast_ets(train_slice, horizon):
    fit = train_ets(train_slice)
    return fit.forecast(horizon).values

def forecast_prophet(train_slice, horizon):
    dates_tr = train_slice.index
    fit = train_prophet(train_slice, dates_tr)
    fut_dates = pd.date_range(start=dates_tr[-1] + pd.Timedelta(days=1), periods=horizon, freq='B')
    fut_df = pd.DataFrame({'ds': fut_dates})
    return fit.predict(fut_df)['yhat'].values

def forecast_state_space(train_slice, horizon):
    fit = train_state_space(train_slice)
    return fit.get_forecast(steps=horizon).predicted_mean.values

def forecast_tree_wrapper(train_slice, horizon, model_type='xgb'):
    # Subset df_ml matching train_slice index
    sub_df = df_ml.iloc[:len(train_slice)].copy()
    X_tr = sub_df[feature_cols]
    y_tr = sub_df['Close']
    
    if model_type == 'xgb':
        m = train_xgboost(X_tr, y_tr)
    elif model_type == 'lgb':
        m = train_lightgbm(X_tr, y_tr)
    else:
        m = train_catboost(X_tr, y_tr)
        
    fc_df = generate_recursive_forecast(m, sub_df, feature_cols, horizon=horizon)
    return fc_df['Predicted_Close'].values

model_fc_funcs = {
    "Naive Baseline": forecast_naive,
    "SARIMA (0,1,0)": forecast_sarima,
    "Exponential Smoothing (ETS)": forecast_ets,
    "Prophet": forecast_prophet,
    "State Space": forecast_state_space,
    "XGBoost": lambda tr, h: forecast_tree_wrapper(tr, h, 'xgb'),
    "LightGBM": lambda tr, h: forecast_tree_wrapper(tr, h, 'lgb'),
    "CatBoost": lambda tr, h: forecast_tree_wrapper(tr, h, 'cat')
}

all_backtest_records = []
print("=== Running Multi-Origin Rolling 30-Day Backtest across 8 Models ===")

for name, fc_func in model_fc_funcs.items():
    print(f"Evaluating model: {name}...")
    res_df = rolling_30day_backtest(
        full_series, fc_func, initial_train_size=initial_train_size, horizon=30, step=30
    )
    res_df['Model_Name'] = name
    all_backtest_records.append(res_df)

combined_bt = pd.concat(all_backtest_records, ignore_index=True)

# Compute Mean and STD across origins
summary_bt = combined_bt.groupby('Model_Name').agg(
    Mean_RMSE=('RMSE', 'mean'),
    Std_RMSE=('RMSE', 'std'),
    Mean_MASE=('MASE', 'mean'),
    Std_MASE=('MASE', 'std'),
    Origins_Count=('RMSE', 'count')
).reset_index()

summary_bt = summary_bt.sort_values('Mean_RMSE').reset_index(drop=True)
print("\n=== Multi-Origin Backtest Summary Leaderboard (Mean ± STD) ===")
print(summary_bt)

# Champion Comparison
single_origin_champ = comparison_df['RMSE'].idxmin()
multi_origin_champ = summary_bt.iloc[0]['Model_Name']

print(f"\nSingle-Origin Champion : {single_origin_champ} (RMSE: {comparison_df.loc[single_origin_champ, 'RMSE']:.2f} INR)")
print(f"Multi-Origin Champion  : {multi_origin_champ} (Mean RMSE: {summary_bt.iloc[0]['Mean_RMSE']:.2f} INR)")

if single_origin_champ == multi_origin_champ:
    print(f"RESULT: Champion model remains unchanged ({single_origin_champ}) under multi-origin backtesting.")
else:
    print(f"ALERT: Champion model shifted from {single_origin_champ} to {multi_origin_champ} under multi-origin backtesting.")

# Export multi-origin backtest summary
summary_bt.to_csv("metrics/multi_origin_backtest.csv", index=False)



## Phase 10: SHAP (SHapley Additive exPlanations) Model Explainability
Generate global summary plots and feature importance rankings using SHAP TreeExplainer.


In [ ]:
import shap

explainer = shap.TreeExplainer(xgb)
shap_values = explainer.shap_values(X_test)

plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_test, show=False)
plt.title("SHAP Feature Importance Summary Plot (XGBoost)", fontweight="bold")
plt.tight_layout()
plt.show()


## Phase 11: Deployment Artifact Export (`export_artifacts.py`)
Refit champion models on full dataset and save deployment artifacts to `models/`, `data/`, and `metrics/`.


In [ ]:
import joblib

MODELS_DIR = "models"
DATA_DIR = "data"
METRICS_DIR = "metrics"

for d in [MODELS_DIR, DATA_DIR, METRICS_DIR]:
    os.makedirs(d, exist_ok=True)

# Refit Champion Models on Full Data
X_full = df_ml[feature_cols]
y_full = df_ml['Close']

xgb_prod = train_xgboost(X_full, y_full)
cat_prod = train_catboost(X_full, y_full)
lgb_prod = train_lightgbm(X_full, y_full)

joblib.dump(xgb_prod, os.path.join(MODELS_DIR, "xgb_model.pkl"))
joblib.dump(cat_prod, os.path.join(MODELS_DIR, "cat_model.pkl"))
joblib.dump(lgb_prod, os.path.join(MODELS_DIR, "lgbm_model.pkl"))
joblib.dump(feature_cols, os.path.join(MODELS_DIR, "feature_cols.pkl"))

# Precompute SHAP values
explainer_full = shap.TreeExplainer(xgb_prod)
shap_vals_full = explainer_full.shap_values(X_full)
joblib.dump(shap_vals_full, os.path.join(MODELS_DIR, "shap_values.pkl"))
joblib.dump(feature_cols, os.path.join(MODELS_DIR, "shap_feature_names.pkl"))

# Save Data and Metrics
df.to_csv(os.path.join(DATA_DIR, "Company_stock_prices_clean.csv"), index=False)
comparison_df.to_csv(os.path.join(METRICS_DIR, "model_comparison.csv"))

print("Successfully exported all deployment artifacts, clean dataset, metrics, and SHAP precomputations.")
